# 07 — Dataprodukt-konsument

Denne notebooken demonstrerer flyten fra oppdagelse til analyse:
1. List tilgjengelige dataprodukter fra portalen
2. Velg et produkt og inspiser metadata
3. Last produktet som pandas DataFrame
4. Sjekk datakvalitet og SLA
5. Enkel analyse

**Forutsetning:** stacken kjører (`docker compose up -d`)

In [ ]:
import sys
sys.path.insert(0, "/home/spark/jobs")

from slettix_client import list_products, get_product, get_manifest, get_quality, get_sla

print("slettix_client lastet ✓")

## 1. Hvilke dataprodukter finnes?

In [ ]:
products = list_products()

print(f"{len(products)} produkt(er) tilgjengelig:\n")
for p in products:
    access = p.get('access', 'public')
    print(f"  {p['id']:30s}  v{p['version']}  [{access}]")
    print(f"    {p['description'][:80]}..." if len(p.get('description','')) > 80 else f"    {p.get('description','')}")

## 2. Inspiser et produkt

In [ ]:
PRODUCT_ID = "hr.employees"   # endre til ønsket produkt

manifest = get_manifest(PRODUCT_ID)

print(f"Navn        : {manifest['name']}")
print(f"Eier        : {manifest['owner']}")
print(f"Domene      : {manifest['domain']}")
print(f"Format      : {manifest['format']}")
print(f"Kildesti    : {manifest['source_path']}")
print(f"Versjon     : {manifest['version']}")
print(f"Tilgang     : {manifest.get('access', 'public')}")
print(f"Tags        : {', '.join(manifest.get('tags', []))}")

if manifest.get('schema'):
    print(f"\nSchema ({len(manifest['schema'])} kolonner):")
    for col in manifest['schema']:
        nullable = '' if col.get('nullable', True) else ' NOT NULL'
        print(f"  {col['name']:20s} {col['type']}{nullable}")

## 3. Last produktet som DataFrame

In [ ]:
df = get_product(PRODUCT_ID)
df.head()

In [ ]:
print(f"Shape   : {df.shape}")
print(f"Kolonner: {list(df.columns)}")
df.dtypes

## 4. Datakvalitet og SLA

In [ ]:
quality = get_quality(PRODUCT_ID)

score = quality.get('score_pct', 0)
icon  = '✓' if score == 100 else ('⚠' if score >= 80 else '✗')

print(f"Kvalitetsscore : {icon} {score}%")
print(f"Forventninger  : {quality.get('passed')}/{quality.get('total_expectations')} bestått")
print(f"Validert       : {quality.get('validated_at', '')[:19].replace('T', ' ')} UTC")

if quality.get('failures'):
    print("\nBrudd:")
    for f in quality['failures']:
        print(f"  - {f['expectation']}")

In [ ]:
sla = get_sla(PRODUCT_ID)

compliant = sla.get('compliant')
icon      = '✓' if compliant else ('✗' if compliant is False else '?')

print(f"SLA-status     : {icon} {'Innenfor SLA' if compliant else 'BRUDD' if compliant is False else 'Ukjent'}")
if sla.get('hours_since_update') is not None:
    print(f"Sist oppdatert : {sla['hours_since_update']}t siden")
    print(f"Krav           : maks {sla.get('freshness_hours')}t")

## 5. Analyse

In [ ]:
# Grunnleggende statistikk
df.describe(include='all')

In [ ]:
# Lønn per avdeling (forutsetter hr.employees-struktur)
if 'department' in df.columns and 'salary' in df.columns:
    summary = (
        df.groupby('department')['salary']
          .agg(['count', 'mean', 'min', 'max'])
          .rename(columns={'count': 'ansatte', 'mean': 'snitt_lønn', 'min': 'min_lønn', 'max': 'maks_lønn'})
          .round(0)
          .sort_values('snitt_lønn', ascending=False)
    )
    print(summary.to_string())
else:
    print("Tilpass analysen til kolonnene i ditt dataprodukt.")